In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("podcastindex_feeds.db")   # use your extracted file's name

pd.read_sql("PRAGMA table_info(podcasts)", conn)          # columns
pd.read_sql("SELECT * FROM podcasts LIMIT 5", conn).T     # sample rows, transposed so columns are readable
pd.read_sql("""SELECT language, COUNT(*) AS n FROM podcasts
               GROUP BY language ORDER BY n DESC LIMIT 30""", conn)

,language,n
0,en,1866288
1,en-us,512043
2,es,461199
3,pt,189134
4,es-ES,148474
5,en-US,140825
6,,115242
7,in,101442
8,fr,100449
9,de,91635


In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 300)

sample = pd.read_sql("SELECT * FROM podcasts WHERE dead = 0 LIMIT 15", conn)
sample.T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
id,1,2,4,5,6,7,8,9,10,11,12,13,14,15,16
url,https://markalanwilliams.libsyn.com/rss,https://anchor.fm/s/19ccb320/podcast/rss,https://feeds.soundcloud.com/users/soundcloud:users:150466516/sounds.rss,https://audioboom.com/channels/4597972.rss,https://anchor.fm/s/212c47d4/podcast/rss,http://feeds.feedburner.com/DirtBikePodcast,https://anchor.fm/s/28f184c/podcast/rss,https://galaktacus.libsyn.com/rss,https://sportsfromherperspective.podomatic.com/rss2.xml,https://feeds.soundcloud.com/users/soundcloud:users:37022259/sounds.rss,https://feeds.soundcloud.com/users/soundcloud:users:151205561/sounds.rss,https://nocturniarecords.podomatic.com/rss2.xml,https://feeds.soundcloud.com/users/soundcloud:users:142613909/sounds.rss,https://noextrawords.libsyn.com/rss,http://feeds.feedburner.com/TheSpysonHour
title,Christianity Questions and Answers,Rahdo Talks Through,IdiotSpeakShow,Conciertos en el Living,OriginALL,2017 U.S. Dirt Bike Events,RNG Show,Galaktacus,Sports From Her Perspective,Romos Music,One Way Ticket to Nilbog,TECHNO TALES OF NOCTURNIA,City Talk w/ Rob Rolison,No Extra Words one person's search for story,The Tyson Hour
lastUpdate,1790336823,1786983470,1776023740,1775938543,1776144276,1717655168,1776153885,1776208256,1776983799,1790924645,1776107975,1775648564,1763234781,1775448893,1773736786
link,http://markalanwilliams.libsyn.com/webpage,https://patreon.com/rahdo,https://soundcloud.com/idiotspeakshow,http://conciertosenelliving.com/podcast/,https://originall.citypodcast.ro,http://www.usdualsports.com/feeds/posts/default,https://podcasters.spotify.com/pod/show/rngshow,http://galaktacus.libsyn.com/podcast,https://www.podomatic.com/podcasts/sportsfromherperspective,https://soundcloud.com/romibee,http://onewaytickettonilbog.weebly.com,https://www.podomatic.com/podcasts/nocturniarecords,https://soundcloud.com/robrolison,http://noextrawords.wordpress.com,https://robhasawebsite.com/
lastHttpStatus,200,200,200,200,200,200,200,200,200,200,200,200,200,200,200
dead,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
contentType,application/rss+xml; charset=utf-8,application/rss+xml; charset=utf-8,application/rss+xml; charset=utf-8,application/rss+xml; charset=utf-8,application/rss+xml; charset=utf-8,text/xml; charset=UTF-8,application/rss+xml; charset=utf-8,application/rss+xml; charset=utf-8,application/xml; charset=utf-8,application/rss+xml; charset=utf-8,application/rss+xml; charset=utf-8,application/xml; charset=utf-8,application/rss+xml; charset=utf-8,application/rss+xml; charset=utf-8,text/xml; charset=UTF-8
itunesId,1000000618,1000016089,1000035657,1000035702,1000035707,1000044791,1000075223,1000081327,1000083229,1000090525,1000090572,1000093513,1000095128,1000095644,1000101673
originalUrl,https://markalanwilliams.libsyn.com/rss,https://anchor.fm/s/19ccb320/podcast/rss,http://feeds.soundcloud.com/users/soundcloud:users:150466516/sounds.rss,https://audioboom.com/channels/4597972.rss,https://anchor.fm/s/212c47d4/podcast/rss,http://feeds.feedburner.com/DirtBikePodcast,https://anchor.fm/s/28f184c/podcast/rss,https://galaktacus.libsyn.com/rss,http://sportsfromherperspective.podomatic.com/rss2.xml,http://feeds.soundcloud.com/users/soundcloud:users:37022259/sounds.rss,http://feeds.soundcloud.com/users/soundcloud:users:151205561/sounds.rss,http://nocturniarecords.podomatic.com/rss2.xml,http://feeds.soundcloud.com/users/soundcloud:users:142613909/sounds.rss,https://noextrawords.libsyn.com/rss,http://feeds.feedburner.com/TheSpysonHour


In [4]:
pd.read_sql("""SELECT category1, COUNT(*) AS n FROM podcasts
               WHERE dead = 0 GROUP BY category1
               ORDER BY n DESC LIMIT 60""", conn)

,category1,n
0,society,787920
1,education,509119
2,business,419229
3,arts,412394
4,religion,398810
5,,286710
6,music,256094
7,health,254897
8,comedy,242575
9,news,240236


In [5]:
main_categories = ["technology", "business"]
topic_terms = ["technology", "startup", "fintech", "founder", "entrepreneur",
               "venture capital", "software", "artificial intelligence", "innovation"]

text = "(title || ' ' || IFNULL(description, ''))"
kw_filter = " OR ".join(f"{text} LIKE '%{t}%'" for t in topic_terms)

query = f"""
SELECT id, podcastGuid, title, description, itunesAuthor, language,
       category1, category2, category3, episodeCount, newestItemPubdate, url
FROM podcasts
WHERE dead = 0
  AND language LIKE 'en%'
  AND episodeCount >= 20
  AND newestItemPubdate >= strftime('%s', '2026-01-01')
  AND (
        category1 IN ('technology', 'business')
        OR ((category1 IS NULL OR category1 = '') AND ({kw_filter}))
      )
"""
candidates = pd.read_sql(query, conn)
len(candidates)

38832

In [6]:
candidates["topic"] = candidates["category1"].replace("", "uncategorized (keyword match)")
candidates["subtopic"] = candidates["category2"].fillna("")
candidates.groupby(["topic", "subtopic"]).size().sort_values(ascending=False).head(20)

topic       subtopic        
business    entrepreneurship    8293
                                7314
            investing           4310
            management          3056
            marketing           3018
            careers             2861
technology                      2681
business    non-profit          1336
            education           1082
technology  business             991
            news                 712
business    technology           564
            news                 541
technology  education            347
business    society              288
technology  science              177
business    health               171
technology  society              150
business    arts                 116
            government            87
dtype: int64

In [7]:
candidates["topic"].value_counts()                      # tech vs business vs uncategorized
candidates["episodeCount"].describe()                   # how big the shows are
candidates.groupby(["category1", "category2"]).size().sort_values(ascending=False).head(15)

category1   category2       
business    entrepreneurship    8293
                                7314
            investing           4310
            management          3056
            marketing           3018
            careers             2861
technology                      2681
business    non-profit          1336
            education           1082
technology  business             991
            news                 712
business    technology           564
            news                 541
technology  education            347
business    society              288
dtype: int64